SwiGLU-6 — long recovery and final evaluation

Load-only reporting for the fixed S5-C2 models at 20% and 50% eligible-MLP removal. Historical prefix metrics and final full-corpus metrics are reported separately. No model loading, recovery, or benchmark execution occurs here.

setup and artifact paths

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "src" / "mlp_replacement").is_dir()
)
RESULTS = PROJECT_ROOT / "data/results/workflows/model/swiglu-6"
RECOVERY_PATHS = [
    RESULTS / "recovery-001-target-0.2.json",
    RESULTS / "recovery-001-target-0.5.json",
]
EVALUATION_PATH = RESULTS / "evaluation-001.json"

recovery_runs = [json.loads(path.read_text(encoding="utf-8")) for path in RECOVERY_PATHS]
evaluation = json.loads(EVALUATION_PATH.read_text(encoding="utf-8"))
for artifact in [*recovery_runs, evaluation]:
    if artifact.get("workflow") != "swiglu-6" or artifact.get("status") != "completed":
        raise ValueError("This final report requires completed SwiGLU-6 artifacts")
pd.set_option("display.max_columns", None)

recovery and historical 100M comparison

In [ ]:
recovery_rows = []
summary_rows = []
for run in recovery_runs:
    results = run["results"]
    summary_rows.append({
        "eligible_mlp_removal": run["target"],
        "tokens": results["tokens_seen"],
        "new_training_hours": results["training_seconds"] / 3600,
        "inherited_search_hours": results["inherited_search_training_seconds"] / 3600,
        "evaluation_hours": results["evaluation_seconds"] / 3600,
        "historical_100m_kl_delta": results["historical_100m_comparison"]["kl_delta"],
        "historical_100m_ppl_delta": results["historical_100m_comparison"]["ppl_delta"],
    })
    for row in results["validation_history"]:
        recovery_rows.append({
            "eligible_mlp_removal": run["target"],
            "cumulative_tokens": row["actual_tokens"],
            "new_training_hours": row["training_seconds"] / 3600,
            "recovery_validation_kl": row["recovery_validation_kl"],
            "legacy_prefix_ppl": row.get("wikitext_validation", {}).get("perplexity"),
        })
summary_df = pd.DataFrame(summary_rows)
recovery_df = pd.DataFrame(recovery_rows)
display(summary_df)

figure, axes = plt.subplots(1, 3, figsize=(16, 4))
for target, rows in recovery_df.groupby("eligible_mlp_removal"):
    rows = rows.sort_values("cumulative_tokens")
    label = f"{target:.0%} eligible-MLP removal"
    axes[0].plot(rows["cumulative_tokens"] / 1e9, rows["recovery_validation_kl"], label=label)
    axes[1].plot(rows["new_training_hours"], rows["recovery_validation_kl"], label=label)
    prefix = rows.dropna(subset=["legacy_prefix_ppl"])
    axes[2].plot(prefix["cumulative_tokens"] / 1e9, prefix["legacy_prefix_ppl"], marker="o", label=label)
axes[0].set(xlabel="Cumulative recovery tokens (B)", ylabel="Historical C4 validation KL")
axes[1].set(xlabel="New training time (GPU hours)", ylabel="Historical C4 validation KL")
axes[2].set(xlabel="Cumulative recovery tokens (B)", ylabel="Historical WikiText prefix PPL")
for axis in axes:
    axis.legend()
figure.tight_layout()
plt.show()

full-corpus BF16 evaluation — validation and test, contexts 128 and 2048

In [ ]:
models = evaluation["results"]["models"]
cohort = {row["id"]: row for row in evaluation["cohort"]}
likelihood_rows = []
footprint_rows = []
for model_id, result in models.items():
    identity = {
        "model": model_id,
        "eligible_mlp_removal": cohort[model_id]["target"],
        "recovery_tokens": cohort[model_id]["tokens"],
    }
    for unit, metrics in result["likelihood"].items():
        likelihood_rows.append({**identity, "split": unit.split("-")[0], **metrics})
    footprint_rows.append({
        **identity,
        **result["footprint"],
        **result["resident_memory"],
        "bf16_conversion_ppl_delta": result.get("bf16_conversion_ppl_delta"),
    })
likelihood_df = pd.DataFrame(likelihood_rows)
footprint_df = pd.DataFrame(footprint_rows)
display(likelihood_df[["model", "split", "context_length", "stride", "predicted_tokens", "loss", "perplexity"]])
display(footprint_df[[
    "model", "parameters", "whole_model_parameter_removal",
    "parameter_bytes", "buffer_bytes", "tensor_file_bytes", "bundle_bytes",
    "gpu_allocated_delta_bytes", "gpu_reserved_delta_bytes",
    "host_rss_delta_bytes", "bf16_conversion_ppl_delta",
]])

full zero-shot tasks and paired differences against dense

In [ ]:
task_rows = []
macro_rows = []
for model_id, comparison in evaluation["results"]["comparisons"].items():
    macro_rows.append({"model": model_id, "macro_accuracy": comparison["macro_accuracy"],
                       "macro_delta": comparison["macro_delta"]})
    for task, row in comparison["tasks"].items():
        task_rows.append({
            "model": model_id, "task": task,
            "primary_metric": evaluation["configuration"]["evaluation"]["primary_metrics"][task],
            "accuracy": row["student"], "dense_accuracy": row["dense"],
            "paired_delta": row["student_minus_dense"],
            "ci95_low": row["ci95"][0], "ci95_high": row["ci95"][1],
            "examples": row["examples"],
        })
task_df = pd.DataFrame(task_rows)
macro_df = pd.DataFrame(macro_rows)
display(task_df)
display(macro_df)

figure, axes = plt.subplots(1, 5, figsize=(19, 5), sharey=True)
student_order = [row["id"] for row in evaluation["cohort"] if row["id"] != "dense"]
for axis, task in zip(axes, evaluation["configuration"]["evaluation"]["tasks"]):
    rows = task_df[task_df["task"] == task].set_index("model").loc[student_order]
    for position, row in enumerate(rows.itertuples()):
        axis.plot([100 * row.ci95_low, 100 * row.ci95_high], [position, position], color="C0")
        axis.plot(100 * row.paired_delta, position, "o", color="C0")
    axis.axvline(0, color="black", linestyle="--")
    axis.set(title=task, xlabel="Difference vs dense (pp)", yticks=range(len(student_order)),
             yticklabels=student_order)
figure.suptitle("95% paired-example bootstrap intervals; no training-seed uncertainty estimate")
figure.tight_layout()
plt.show()

quality and measured deployment footprint

In [ ]:
quality_df = (
    likelihood_df[(likelihood_df["split"] == "test") & (likelihood_df["context_length"] == 2048)]
    .merge(footprint_df[["model", "tensor_file_bytes", "gpu_allocated_delta_bytes"]], on="model", validate="one_to_one")
    .merge(macro_df, on="model", validate="one_to_one")
)
figure, axes = plt.subplots(1, 2, figsize=(13, 5))
for row in quality_df.itertuples():
    axes[0].scatter(row.tensor_file_bytes / 2**30, row.perplexity)
    axes[0].annotate(row.model, (row.tensor_file_bytes / 2**30, row.perplexity), fontsize=8)
    axes[1].scatter(row.gpu_allocated_delta_bytes / 2**30, 100 * row.macro_accuracy)
    axes[1].annotate(row.model, (row.gpu_allocated_delta_bytes / 2**30, 100 * row.macro_accuracy), fontsize=8)
axes[0].set(xlabel="BF16 tensor file size (GiB)", ylabel="Full WikiText test PPL (context 2048)")
axes[1].set(xlabel="Resident GPU allocated delta (GiB)", ylabel="Macro task accuracy (%)")
figure.tight_layout()
plt.show()

provenance and limits

Removal targets refer to eligible MLP parameters; the footprint table separately reports whole-model removal. Resident memory comes from fresh processes with no forward pass. This report does not establish serving latency or multi-seed recovery uncertainty. S5-C2 is the historical selected allocation, with the SwiGLU-5 selection-label caveat retained in the run guide.

In [ ]:
display(pd.DataFrame([
    {"artifact": str(path.relative_to(PROJECT_ROOT)),
     "fingerprint": run["run_fingerprint"],
     "prepared_sha256": run["prepared_sha256"],
     "status": run["status"]}
    for path, run in zip(RECOVERY_PATHS, recovery_runs)
]))
display({
    "evaluation_fingerprint": evaluation["evaluation_fingerprint"],
    "protocol_sha256": evaluation["protocol_sha256"],
    "environment": evaluation["environment"],
})